In [ ]:
!pip install timm -q

import torch
import torch.nn as nn
import timm
import pandas as pd
import numpy as np
import json
import os
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import gaussian_kde
from sklearn.metrics import (
    confusion_matrix, roc_auc_score,
    f1_score, accuracy_score,
    roc_curve, auc as sklearn_auc
)
from torchvision import transforms, models
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from google.colab import drive

drive.mount('/content/drive')

BASE          = '/content/drive/MyDrive/CDT_Dataset'
HUMAN_FOLDER  = f'{BASE}/images_human_fixed'
AUG_FOLDER    = f'{BASE}/images_human_aug'
RESULT_FOLDER = f'{BASE}/results'
FEATURE_PATH  = f'{BASE}/labels/vqa_features.json'

os.makedirs(RESULT_FOLDER, exist_ok=True)

device = torch.device(
    'cuda' if torch.cuda.is_available() else 'cpu'
)
CUTOFF = 6

test_df  = pd.read_csv(
    f'{BASE}/labels/human_test.csv'
)
train_df = pd.read_csv(
    f'{BASE}/labels/human_train.csv'
)
val_df   = pd.read_csv(
    f'{BASE}/labels/human_val.csv'
)
for d in [test_df, train_df, val_df]:
    d['dementia_risk'] = d['dementia_risk'].astype(bool)

with open(FEATURE_PATH) as f:
    vqa_features = json.load(f)

def get_vqa_feat(filename, vqa_dict):
    if filename in vqa_dict:
        return vqa_dict[filename]
    alt = filename.replace('.jpg','.tif') \
          if '.jpg' in filename \
          else filename.replace('.tif','.jpg')
    return vqa_dict.get(alt, {})

print(f'Device: {device}')
print(f'Test  : {len(test_df)} รูป')

In [ ]:
class CDTDataset(Dataset):
    def __init__(self, df, image_folder,
                 aug_folder=None, transform=None):
        self.df           = df.reset_index(drop=True)
        self.image_folder = image_folder
        self.aug_folder   = aug_folder
        self.transform    = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        filename = row['filename']
        if (self.aug_folder and
                filename.startswith('aug')):
            img_path = f'{self.aug_folder}/{filename}'
        else:
            img_path = f'{self.image_folder}/{filename}'
        try:
            img = Image.open(img_path).convert('RGB')
        except:
            img = Image.new('RGB',(224,224),(255,255,255))
        if self.transform:
            img = self.transform(img)
        labels = torch.tensor([
            row['domain_A'], row['domain_B'],
            row['domain_C'], row['domain_D'],
            row['domain_E']
        ], dtype=torch.float32)
        return img, labels

class CDTHybridDataset(Dataset):
    def __init__(self, df, image_folder,
                 vqa_feats, aug_folder=None,
                 transform=None):
        self.df           = df.reset_index(drop=True)
        self.image_folder = image_folder
        self.vqa_feats    = vqa_feats
        self.aug_folder   = aug_folder
        self.transform    = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        filename = row['filename']
        if (self.aug_folder and
                filename.startswith('aug')):
            img_path = f'{self.aug_folder}/{filename}'
        else:
            img_path = f'{self.image_folder}/{filename}'
        try:
            img = Image.open(img_path).convert('RGB')
        except:
            img = Image.new('RGB',(224,224),(255,255,255))
        if self.transform:
            img = self.transform(img)
        feat = torch.tensor(
            features_to_vector(
                get_vqa_feat(filename, self.vqa_feats)
            ), dtype=torch.float32
        )
        labels = torch.tensor([
            row['domain_A'], row['domain_B'],
            row['domain_C'], row['domain_D'],
            row['domain_E']
        ], dtype=torch.float32)
        return img, feat, labels

class CDTModel(nn.Module):
    def __init__(self, backbone_name='vit',
                 num_domains=5, num_classes=3):
        super().__init__()
        if backbone_name == 'vit':
            self.backbone = timm.create_model(
                'vit_base_patch16_224',
                pretrained=True, num_classes=0
            )
            self.feature_dim = 768
        else:
            backbone = models.resnet50(
                weights=models.ResNet50_Weights.IMAGENET1K_V1
            )
            self.feature_dim = 2048
            self.backbone = nn.Sequential(
                *list(backbone.children())[:-1]
            )
        self.backbone_name = backbone_name
        for p in self.backbone.parameters():
            p.requires_grad = False
        self.shared = nn.Sequential(
            nn.Flatten(),
            nn.Linear(self.feature_dim, 512),
            nn.ReLU(), nn.Dropout(0.3)
        )
        self.heads_reg = nn.ModuleList([
            nn.Linear(512, 1)
            for _ in range(num_domains)
        ])
        self.heads_cls = nn.ModuleList([
            nn.Linear(512, num_classes)
            for _ in range(num_domains)
        ])

    def forward(self, x, mode='reg'):
        if self.backbone_name == 'vit':
            feat = self.backbone(x)
            feat = feat.unsqueeze(-1).unsqueeze(-1)
        else:
            feat = self.backbone(x)
        feat = self.shared(feat)
        if mode == 'reg':
            return [h(feat).squeeze(-1)
                    for h in self.heads_reg]
        return [h(feat) for h in self.heads_cls]

class CDTHybridModel(nn.Module):
    def __init__(self, backbone_name='vit',
                 vqa_dim=12, num_domains=5):
        super().__init__()
        if backbone_name == 'vit':
            self.backbone = timm.create_model(
                'vit_base_patch16_224',
                pretrained=True, num_classes=0
            )
            vis_dim = 768
        else:
            backbone = models.resnet50(
                weights=models.ResNet50_Weights.IMAGENET1K_V1
            )
            self.backbone = nn.Sequential(
                *list(backbone.children())[:-1]
            )
            vis_dim = 2048
        self.backbone_name = backbone_name
        for p in self.backbone.parameters():
            p.requires_grad = False
        self.vis_shared = nn.Sequential(
            nn.Flatten(),
            nn.Linear(vis_dim, 512),
            nn.ReLU(), nn.Dropout(0.3)
        )
        self.vqa_encoder = nn.Sequential(
            nn.Linear(vqa_dim, 64),
            nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 32), nn.ReLU()
        )
        self.fusion = nn.Sequential(
            nn.Linear(512+32, 256),
            nn.ReLU(), nn.Dropout(0.3)
        )
        self.heads_reg = nn.ModuleList([
            nn.Linear(256, 1)
            for _ in range(num_domains)
        ])
        self.heads_cls = nn.ModuleList([
            nn.Linear(256, 3)
            for _ in range(num_domains)
        ])

    def forward(self, img, vqa_feat, mode='reg'):
        if self.backbone_name == 'vit':
            vis = self.backbone(img)
            vis = vis.unsqueeze(-1).unsqueeze(-1)
        else:
            vis = self.backbone(img)
        vis   = self.vis_shared(vis)
        vqa   = self.vqa_encoder(vqa_feat)
        fused = self.fusion(
            torch.cat([vis, vqa], dim=1)
        )
        if mode == 'reg':
            return [h(fused).squeeze(-1)
                    for h in self.heads_reg]
        return [h(fused) for h in self.heads_cls]

def features_to_vector(features):
    if features is None or 'error' in features:
        return np.zeros(12)
    vec = []
    vec.append(features.get('digit_count', 0) / 12.0)
    digits = features.get('digits_present', [])
    vec.append(len(digits) / 12.0)
    vec.append(1.0 if features.get(
        'digit_positions_correct') == 'Yes' else 0.0)
    vec.append(
        features.get('quadrant_balance', 0) / 4.0)
    wq = features.get('worst_quadrant', 'none')
    wq_map = {
        'top-left':[1,0,0,0,0], 'top-right':[0,1,0,0,0],
        'bottom-left':[0,0,1,0,0],
        'bottom-right':[0,0,0,1,0], 'none':[0,0,0,0,1],
    }
    vec.extend(wq_map.get(wq, [0,0,0,0,1]))
    vec.append(1.0 if features.get(
        'has_hands') == 'Yes' else 0.0)
    vec.append(features.get('hand_count', 0) / 2.0)
    vec.append(1.0 if features.get(
        'hands_at_correct_position') == 'Yes' else 0.0)
    vec.append(
        features.get('spatial_regularity', 1) / 5.0)
    vec.append(
        features.get('overall_quality', 1) / 5.0)
    return np.array(vec, dtype=np.float32)

def get_predictions(outputs, mode='reg'):
    preds = []
    for output in outputs:
        if mode == 'reg':
            pred = output.round().clamp(0,2).long()
        else:
            pred = output.argmax(dim=1)
        preds.append(pred)
    return torch.stack(preds, dim=1)

def get_raw_scores(outputs, mode='reg'):
    if mode == 'reg':
        return torch.stack(
            [o.squeeze(-1) for o in outputs], dim=1
        )
    return torch.stack([
        (torch.softmax(o, dim=1) *
         torch.tensor([0.,1.,2.]).to(device)
        ).sum(dim=1)
        for o in outputs
    ], dim=1)

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std =[0.229, 0.224, 0.225]
    )
])

domain_names = {
    0: 'A: digit_count',
    1: 'B: worst_quadrant',
    2: 'C: spatial',
    3: 'D: hands_present',
    4: 'E: hands_placement',
}
colors_true = {0:'#3F51B5', 1:'#00BCD4', 2:'#8BC34A'}

In [ ]:
import torch

for name, path in [
    ('07l hybrid_vit_reg',
     f'{BASE}/models_hybrid/hybrid_vit_reg_best.pth'),
    ('07l hybrid_vit_reg_new',
     f'{BASE}/models_hybrid/hybrid_vit_reg_new_best.pth'),
]:
    if not os.path.exists(path):
        print(f'⚠️ ไม่พบ {name}')
        continue
    state = torch.load(path, map_location='cpu')
    dim   = state['vqa_encoder.0.weight'].shape[1]
    print(f'{name}: vqa_dim = {dim}')

In [ ]:
def evaluate_model_full(model, loader,
                        mode='reg', has_vqa=False):
    model.eval()
    all_preds  = []
    all_labels = []
    all_raw    = []

    with torch.no_grad():
        for batch in loader:
            if has_vqa:
                images, feat, labels = batch
                feat    = feat.to(device)
            else:
                images, labels = batch
                feat    = None

            images  = images.to(device)

            if feat is not None:
                outputs = model(images, feat, mode)
            else:
                outputs = model(images, mode=mode)

            preds = get_predictions(outputs, mode)
            raw   = get_raw_scores(outputs, mode)

            all_preds.append(preds.cpu())
            all_labels.append(labels.cpu())
            all_raw.append(raw.cpu())

    preds_arr  = torch.cat(all_preds).numpy()
    labels_arr = torch.cat(all_labels).numpy()
    raw_arr    = torch.cat(all_raw).numpy()

    pred_total    = preds_arr.sum(axis=1)
    true_total    = labels_arr.sum(axis=1)
    pred_dementia = (pred_total < CUTOFF).astype(int)
    true_dementia = (true_total < CUTOFF).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        true_dementia, pred_dementia, labels=[0,1]
    ).ravel()

    sens = tp/(tp+fn) if (tp+fn) > 0 else 0
    spec = tn/(tn+fp) if (tn+fp) > 0 else 0
    try:
        auc = roc_auc_score(true_dementia, -pred_total)
        fpr, tpr, _ = roc_curve(
            true_dementia, -pred_total
        )
    except:
        auc, fpr, tpr = 0, [], []

    domain_acc = {}
    for i, d in enumerate(['A','B','C','D','E']):
        true_d = labels_arr[:, i].astype(int)
        pred_d = preds_arr[:, i].astype(int)
        domain_acc[d] = (true_d == pred_d).mean()

    return {
        'sensitivity'  : sens,
        'specificity'  : spec,
        'auc'          : auc,
        'fpr'          : fpr,
        'tpr'          : tpr,
        'domain_acc'   : domain_acc,
        'preds_arr'    : preds_arr,
        'labels_arr'   : labels_arr,
        'raw_arr'      : raw_arr,
        'pred_total'   : pred_total,
        'true_dementia': true_dementia,
    }

models_config = [
    {
        'name'     : '07k ViT REG',
        'path'     : f'{BASE}/models_human_undersample/human_us_vit_reg_best.pth',
        'class'    : 'CDTModel',
        'backbone' : 'vit',
        'mode'     : 'reg',
        'has_vqa'  : False,
        'color'    : '#9E9E9E',
    },
    {
        'name'     : '07l Hybrid VQA',
        'path'     : f'{BASE}/models_hybrid/hybrid_vit_reg_best.pth',
        'class'    : 'CDTHybridModel',
        'backbone' : 'vit',
        'mode'     : 'reg',
        'has_vqa'  : True,
        'vqa_dim'  : 14,
        'color'    : '#2196F3',
    },
    {
        'name'     : '07r Focal g=2.0',
        'path'     : f'{BASE}/models_focal/focal_vit_cls_g20_best.pth',
        'class'    : 'CDTModel',
        'backbone' : 'vit',
        'mode'     : 'cls',
        'has_vqa'  : False,
        'color'    : '#FF9800',
    },
]

test_ds_plain = CDTDataset(
    test_df, HUMAN_FOLDER, None, val_transform
)
test_loader_plain = DataLoader(
    test_ds_plain, batch_size=16, shuffle=False
)

test_ds_vqa = CDTHybridDataset(
    test_df, HUMAN_FOLDER,
    vqa_features, None, val_transform
)
test_loader_vqa = DataLoader(
    test_ds_vqa, batch_size=16, shuffle=False
)

all_results = {}

for cfg in models_config:
    name = cfg['name']
    path = cfg['path']

    if not os.path.exists(path):
        print(f'ไม่พบ {name}')
        continue

    if cfg['class'] == 'CDTModel':
        model = CDTModel(
            backbone_name=cfg['backbone']
        ).to(device)
    else:
        model = CDTHybridModel(
            backbone_name=cfg['backbone'],
            vqa_dim=cfg.get('vqa_dim', 14)
        ).to(device)

    model.load_state_dict(
        torch.load(path, map_location=device)
    )
    model.eval()

    loader = (test_loader_vqa
              if cfg['has_vqa']
              else test_loader_plain)

    results = evaluate_model_full(
        model, loader,
        mode=cfg['mode'],
        has_vqa=cfg['has_vqa']
    )
    results['color'] = cfg['color']
    all_results[name] = results

    print(f'{name}: '
          f'Sens={results["sensitivity"]:.3f} '
          f'Spec={results["specificity"]:.3f} '
          f'AUC={results["auc"]:.3f}')

extra_results = {
    '07n Downsample': {
        'sensitivity': 0.791, 'specificity': 0.814,
        'auc': 0.880, 'color': '#4CAF50',
        'fpr': None, 'tpr': None,
    },
    '07o Binary': {
        'sensitivity': 0.816, 'specificity': 0.823,
        'auc': 0.866, 'color': '#9C27B0',
        'fpr': None, 'tpr': None,
    },
}

In [ ]:
combined = {**all_results, **extra_results}

print('='*75)
print('FINAL EVALUATION — CDT Dementia Screening')
print('='*75)
print(f'{"Model":<25} {"Sens":>8} '
      f'{"Spec":>8} {"AUC":>8} '
      f'{"Sens✅":>7} {"Spec✅":>7}')
print('='*75)

best_auc = max(
    v['auc'] for v in combined.values()
)

for name, r in combined.items():
    sens = r['sensitivity']
    spec = r['specificity']
    auc_v= r['auc']
    mark = '✅' if auc_v == best_auc else ''
    goal = '🎯' if sens>=0.88 and spec>=0.82 else ''
    s_ok = '✅' if sens >= 0.88 else '❌'
    p_ok = '✅' if spec >= 0.82 else '❌'

    print(f'{name:<25} '
          f'{sens:>8.3f} '
          f'{spec:>8.3f} '
          f'{auc_v:>8.3f} '
          f'{s_ok:>7} {p_ok:>7} {mark}{goal}')

print('='*75)
print(f'{"Target":<25} {"≥0.880":>8} '
      f'{"≥0.820":>8} {"≥0.910":>8}')
print()

best_name = max(
    all_results.keys(),
    key=lambda k: all_results[k]['auc']
)
best_r = all_results[best_name]

print(f'Cutoff Analysis — {best_name}:')
print(f'{"Cutoff":>8} {"Sens":>8} '
      f'{"Spec":>8} {"F1":>8}')
print('-'*38)

for cutoff in [4, 5, 6, 7, 8]:
    pred = (
        best_r['pred_total'] < cutoff
    ).astype(int)
    try:
        tn2, fp2, fn2, tp2 = confusion_matrix(
            best_r['true_dementia'],
            pred, labels=[0,1]
        ).ravel()
        s  = tp2/(tp2+fn2) if (tp2+fn2)>0 else 0
        sp = tn2/(tn2+fp2) if (tn2+fp2)>0 else 0
        f1 = f1_score(best_r['true_dementia'], pred)
    except:
        s, sp, f1 = 0, 0, 0
    mark = '🎯' if s>=0.88 and sp>=0.82 else ''
    print(f'{cutoff:>8} {s:>8.3f} '
          f'{sp:>8.3f} {f1:>8.3f} {mark}')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))

for name, r in all_results.items():
    if r.get('fpr') is not None and len(
        r['fpr']
    ) > 0:
        ax.plot(
            r['fpr'], r['tpr'],
            color=r['color'],
            linewidth=2,
            label=f'{name} (AUC={r["auc"]:.3f})'
        )

ax.plot([0,1],[0,1], 'k--', alpha=0.5,
        label='Random')
ax.axhline(y=0.88, color='red', linestyle=':',
           alpha=0.7, label='Sens target 0.88')
ax.axvline(x=1-0.82, color='blue', linestyle=':',
           alpha=0.7, label='Spec target 0.82')

ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title(
    'ROC Curve — All Models\n'
    'CDT Dementia Screening',
    fontsize=13, fontweight='bold'
)
ax.legend(fontsize=9, loc='lower right')
ax.grid(alpha=0.3)
ax.set_xlim([-0.02, 1.02])
ax.set_ylim([-0.02, 1.02])

plt.tight_layout()
plt.savefig(
    f'{RESULT_FOLDER}/08_roc_curve.png',
    dpi=150, bbox_inches='tight'
)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 7))

names  = list(combined.keys())
colors = [combined[n]['color'] for n in names]

for ax_idx, (metric, title, target) in enumerate([
    ('sensitivity', 'Sensitivity', 0.88),
    ('specificity', 'Specificity', 0.82),
    ('auc',         'AUC-ROC',     0.91),
]):
    vals = [combined[n][metric] for n in names]
    bars = axes[ax_idx].barh(
        names, vals, color=colors,
        edgecolor='white', height=0.6
    )
    axes[ax_idx].axvline(
        x=target, color='red',
        linestyle='--', linewidth=2,
        label=f'Target {target}'
    )
    axes[ax_idx].set_title(
        title, fontsize=12, fontweight='bold'
    )
    axes[ax_idx].set_xlim(0.5, 1.05)
    axes[ax_idx].legend(fontsize=9)

    for bar, val in zip(bars, vals):
        color_text = 'green' if (
            (metric == 'sensitivity' and val >= 0.88) or
            (metric == 'specificity' and val >= 0.82) or
            (metric == 'auc' and val >= 0.91)
        ) else 'black'
        axes[ax_idx].text(
            val + 0.003,
            bar.get_y() + bar.get_height()/2,
            f'{val:.3f}',
            va='center', fontsize=9,
            color=color_text,
            fontweight='bold'
        )

plt.suptitle(
    'Model Comparison — CDT Dementia Screening\n'
    'All Experiments',
    fontsize=14, fontweight='bold'
)
plt.tight_layout()
plt.savefig(
    f'{RESULT_FOLDER}/08_model_comparison.png',
    dpi=150, bbox_inches='tight'
)
plt.show()

In [ ]:
best_r = all_results[best_name]

fig, axes = plt.subplots(2, 5, figsize=(22, 9))

for d_idx, d_name in domain_names.items():
    true_d = best_r['labels_arr'][:,d_idx].astype(int)
    pred_d = best_r['preds_arr'][:,d_idx].astype(int)

    cm      = confusion_matrix(
        true_d, pred_d, labels=[0,1,2]
    )
    cm_norm = cm.astype(float)
    rs      = cm.sum(axis=1, keepdims=True)
    rs[rs==0] = 1
    cm_norm = cm_norm / rs

    for row_idx, (data, fmt, sfx) in enumerate([
        (cm,      'd',   ''),
        (cm_norm, '.2f', '(normalized)')
    ]):
        ax = axes[row_idx][d_idx]
        sns.heatmap(
            data, annot=True, fmt=fmt,
            cmap='Blues', ax=ax,
            xticklabels=['0','1','2'],
            yticklabels=['0','1','2'],
            cbar=False
        )
        acc = (true_d == pred_d).mean()
        ax.set_title(
            f'{d_name}\nAcc={acc:.3f} {sfx}',
            fontsize=8, fontweight='bold'
        )
        ax.set_xlabel('Predicted')
        ax.set_ylabel('True')

plt.suptitle(
    f'Confusion Matrix — {best_name} (Best Model)\n'
    f'Sens={best_r["sensitivity"]:.3f} '
    f'Spec={best_r["specificity"]:.3f} '
    f'AUC={best_r["auc"]:.3f}',
    fontsize=12, fontweight='bold'
)
plt.tight_layout()
plt.savefig(
    f'{RESULT_FOLDER}/08_best_confusion_matrix.png',
    dpi=150, bbox_inches='tight'
)
plt.show()

In [ ]:
n_models = len(all_results)
fig, axes = plt.subplots(
    n_models, 5,
    figsize=(25, n_models * 4)
)

for row_idx, (name, r) in enumerate(
    all_results.items()
):
    for d_idx, d_name in domain_names.items():
        ax      = axes[row_idx][d_idx] \
                  if n_models > 1 \
                  else axes[d_idx]
        true_d  = r['labels_arr'][:,d_idx].astype(int)
        score_d = r['raw_arr'][:,d_idx]

        for true_val in [0, 1, 2]:
            mask   = (true_d == true_val)
            scores = score_d[mask]
            n      = mask.sum()
            if n < 2:
                continue
            try:
                kde    = gaussian_kde(
                    scores, bw_method=0.3
                )
                x_vals = np.linspace(
                    scores.min()-0.2,
                    scores.max()+0.2, 300
                )
                ax.plot(
                    x_vals, kde(x_vals),
                    color=colors_true[true_val],
                    linewidth=2,
                    label=f'true={true_val} (n={n})'
                )
                ax.fill_between(
                    x_vals, kde(x_vals),
                    color=colors_true[true_val],
                    alpha=0.15
                )
            except:
                pass

        for x in [0.5, 1.5]:
            ax.axvline(x=x, color='gray',
                      linestyle='--', alpha=0.4)

        title = f'{name}\n{d_name}' \
                if d_idx == 0 else d_name
        ax.set_title(title, fontsize=8,
                     fontweight='bold')
        ax.set_xlabel('Score', fontsize=7)
        ax.set_ylabel('Density', fontsize=7)

        if d_idx == 0 and row_idx == 0:
            ax.legend(fontsize=6)

plt.suptitle(
    'Score Distribution — All Models\n'
    'blue=0  cyan=1  green=2\n'
    'Separated peaks = better separation',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.savefig(
    f'{RESULT_FOLDER}/08_distribution_all.png',
    dpi=150, bbox_inches='tight'
)
plt.show()

In [ ]:
domain_keys = ['A', 'B', 'C', 'D', 'E']
model_names = list(all_results.keys())

acc_matrix = np.zeros(
    (len(model_names), len(domain_keys))
)

for i, name in enumerate(model_names):
    for j, d in enumerate(domain_keys):
        acc_matrix[i][j] = all_results[name][
            'domain_acc'
        ].get(d, 0)

fig, ax = plt.subplots(figsize=(10, 6))

sns.heatmap(
    acc_matrix,
    annot=True, fmt='.3f',
    cmap='RdYlGn',
    xticklabels=[
        f'{d}\n{list(domain_names.values())[i].split(":")[1].strip()}'
        for i, d in enumerate(domain_keys)
    ],
    yticklabels=model_names,
    vmin=0.3, vmax=0.9,
    linewidths=0.5,
    ax=ax
)
ax.set_title(
    'Per-Domain Accuracy — All Models\n',
    fontsize=12, fontweight='bold'
)
ax.set_xlabel('Domain')
ax.set_ylabel('Model')

plt.tight_layout()
plt.savefig(
    f'{RESULT_FOLDER}/08_domain_accuracy_heatmap.png',
    dpi=150, bbox_inches='tight'
)
plt.show()

In [ ]:
final_summary = {}
for name, r in combined.items():
    final_summary[name] = {
        'sensitivity': float(r['sensitivity']),
        'specificity': float(r['specificity']),
        'auc'        : float(r['auc']),
    }
    if 'domain_acc' in r:
        final_summary[name]['domain_acc'] = {
            k: float(v)
            for k, v in r['domain_acc'].items()
        }

with open(
    f'{RESULT_FOLDER}/08_final_summary.json', 'w'
) as f:
    json.dump(final_summary, f, indent=2)

print('='*65)
print('FINAL SUMMARY')
print('='*65)
print(f'Dataset    : NHATS Round 14 CDT')
print(f'Test set   : {len(test_df)} รูป')
print(f'Scoring    : CCSS (0/1/2 per domain)')
print(f'Cutoff     : total score < 6 = dementia')
print()
print(f'Best Model : {best_name}')
print(f'  AUC      : {all_results[best_name]["auc"]:.3f}')
print(f'  Sens     : {all_results[best_name]["sensitivity"]:.3f}')
print(f'  Spec     : {all_results[best_name]["specificity"]:.3f}')